In [8]:
import os
import json
import time
import random
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy import Translator
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

import os, re, json, time, logging
from datasets import get_dataset_config_names, load_dataset, Dataset, DatasetDict
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# Создаём переводчик
yandex = YandexTranslate()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

In [ ]:
from datasets import get_dataset_config_names, load_dataset

REPO_ID = "DeepPavlov/iKAT_2023"

print("Доступные конфигурации:", get_dataset_config_names(REPO_ID))

for config in get_dataset_config_names(REPO_ID):
    print(f"\n{'='*50}")
    print(f"Конфигурация: {config}")
    ds = load_dataset(REPO_ID, config)
    for split in ds:
        print(f"  Сплит: {split}, размер: {len(ds[split])}")
        print(f"    Колонки: {ds[split].column_names}")
        print(f"    Пример: {ds[split][0]}")

In [4]:
# -------------------- Настройки --------------------
SOURCE_REPO_ID = "DeepPavlov/iKAT_2023"
LOCAL_SAVE_PATH = "./iKAT_2023_ru"
CACHE_FILE = "translation_cache_ikat.jsonl"

In [5]:
# -------------------- Кэш --------------------
def load_cache():
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    return cache

def append_cache(text, translation):
    with open(CACHE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")

translation_cache = load_cache()

In [6]:
# -------------------- Улучшенный переводчик --------------------
def is_numeric_string(s: str) -> bool:
    return not bool(re.search(r'[A-Za-zА-Яа-яёЁ]', s))

def translate_text_robust(text, retries=3, delay=3):
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    if is_numeric_string(text):
        return text, True
    if text in translation_cache:
        return translation_cache[text], True

    # Очень длинный (>20000) → абзацы
    if len(text) > 20000:
        logging.info(f"Extremely long ({len(text)} chars), paragraphs...")
        paragraphs = re.split(r'\n\s*\n', text)
        if len(paragraphs) > 1:
            translated_paragraphs, ok = [], True
            for p in paragraphs:
                if not p.strip():
                    translated_paragraphs.append(p); continue
                t, o = translate_text_robust(p, retries, delay)
                if not o: ok = False; break
                translated_paragraphs.append(t)
            if ok:
                full = '\n\n'.join(translated_paragraphs)
                translation_cache[text] = full
                append_cache(text, full)
                return full, True

    # Длинный (>8000) → предложения
    if len(text) > 8000:
        logging.info(f"Long ({len(text)} chars), sentences...")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) > 1:
            translated_parts, ok = [], True
            for s in sentences:
                t, o = translate_text_robust(s, retries, delay)
                if not o: ok = False; break
                translated_parts.append(t)
            if ok:
                full = ' '.join(translated_parts)
                translation_cache[text] = full
                append_cache(text, full)
                return full, True

    last_exception = None
    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            last_exception = e
            error_str = str(e).lower()
            if '413' in error_str or 'too long' in error_str:
                logging.info(f"413, splitting...")
                sentences = re.split(r'(?<=[.!?])\s+', text)
                if len(sentences) > 1:
                    parts, ok = [], True
                    for s in sentences:
                        t, o = translate_text_robust(s, 1, delay)
                        if not o: ok = False; break
                        parts.append(t)
                    if ok:
                        full = ' '.join(parts)
                        translation_cache[text] = full
                        append_cache(text, full)
                        return full, True
            if any(code in error_str for code in ['502','503','504']):
                time.sleep(delay*(attempt+1))
            elif '429' in error_str:
                time.sleep(delay*4+10)
            else:
                time.sleep(delay)

    # Последний шанс для длинного текста
    if len(text) > 2000:
        logging.info("Final split attempt...")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) > 1:
            parts, ok = [], True
            for s in sentences:
                t, o = translate_text_robust(s, 1, delay)
                if not o: ok = False; break
                parts.append(t)
            if ok:
                full = ' '.join(parts)
                translation_cache[text] = full
                append_cache(text, full)
                return full, True

    logging.error(f"Failed: '{text[:50]}...' Error: {last_exception}")
    return "", False

# -------------------- Определение текстовых колонок --------------------
def get_text_columns(dataset):
    """Возвращает список колонок, которые содержат строки и требуют перевода."""
    text_cols = []
    for col in dataset.column_names:
        # Проверяем на небольшом количестве примеров, что колонка содержит строки
        if col.startswith('_'):   # служебные id обычно не переводим
            continue
        sample = dataset[0][col]
        if isinstance(sample, str):
            text_cols.append(col)
        elif isinstance(sample, list) and all(isinstance(item, str) for item in sample):
            text_cols.append(col)
    return text_cols

# -------------------- Перевод одного примера --------------------
def translate_example(example, text_columns):
    result = dict(example)
    success = True
    for col in text_columns:
        value = example[col]
        if isinstance(value, str):
            trans, ok = translate_text_robust(value)
            if not ok: success = False
            result[col + '_ru'] = trans
        elif isinstance(value, list):
            trans_list, col_ok = [], True
            for item in value:
                t, o = translate_text_robust(item)
                if not o: col_ok = False
                trans_list.append(t)
            if not col_ok: success = False
            result[col + '_ru'] = trans_list
        else:
            result[col + '_ru'] = value   # не переводим
    result['_success'] = success
    return result

# -------------------- Обработка сплита --------------------
def process_config_split(config_name, split_name, source_split, progress_file):
    translated_records = []
    failed_indices = set()

    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if rec.get('_failed', False):
                        failed_indices.add(rec['_index'])
                    else:
                        translated_records.append(rec)
                except:
                    continue
        logging.info(f"[{config_name}/{split_name}] Resuming: {len(translated_records)} ok, {len(failed_indices)} failed")

    text_cols = get_text_columns(source_split)
    if not text_cols:
        logging.info(f"[{config_name}/{split_name}] No text columns, copying original.")
        # Просто копируем датасет без перевода
        return source_split

    start_index = len(translated_records) + len(failed_indices)
    total = len(source_split)

    if start_index < total:
        logging.info(f"[{config_name}/{split_name}] Starting from index {start_index}...")
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_split.select(range(start_index, total))),
                desc=f"Translating {config_name}/{split_name}",
                total=total - start_index
            )
            for idx, example in pbar:
                global_idx = start_index + idx
                if global_idx in {r.get('_index', -1) for r in translated_records}:
                    continue

                translated = translate_example(example, text_cols)
                record_out = {'_index': global_idx, '_failed': not translated['_success'], **translated}
                del record_out['_success']

                f.write(json.dumps(record_out, ensure_ascii=False) + "\n")
                f.flush()
                time.sleep(0.5)

                if not record_out['_failed']:
                    translated_records.append(record_out)
                else:
                    failed_indices.add(global_idx)

    # Сбор успешных
    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if not rec.get('_failed', False):
                        rec.pop('_index', None)
                        rec.pop('_failed', None)
                        all_successful.append(rec)
                except:
                    continue

    if not all_successful:
        logging.error(f"[{config_name}/{split_name}] No successful records!")
        return None

    ok_cnt = len(all_successful)
    fail_cnt = total - ok_cnt
    logging.info(f"[{config_name}/{split_name}] Done: {ok_cnt}/{total} ({fail_cnt} failed)")
    return Dataset.from_list(all_successful)

In [ ]:
# -------------------- Главный запуск --------------------
logging.info("Loading iKAT_2023...")
configs = get_dataset_config_names(SOURCE_REPO_ID)
print(f"Конфигурации: {configs}")

all_data = {}
for config in configs:
    source = load_dataset(SOURCE_REPO_ID, config)

    # qrels не переводим вообще – просто копируем оригинал
    if config == 'qrels':
        all_data[config] = source
        print(f"[qrels] Скопирован без перевода")
        continue

    config_splits = {}
    for split in source:
        progress_file = f"translated_ikat_{config}_{split}.jsonl"
        ds_translated = process_config_split(config, split, source[split], progress_file)
        if ds_translated is not None:
            config_splits[split] = ds_translated
    if config_splits:
        all_data[config] = DatasetDict(config_splits)

# Сохранение
if all_data:
    final_dataset = DatasetDict(all_data)
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"\nСохранено в {LOCAL_SAVE_PATH}")

    # Сравнение с оригиналом
    print("\n" + "="*60)
    print("СРАВНЕНИЕ С ОРИГИНАЛОМ")
    print("="*60)
    for config in configs:
        orig = load_dataset(SOURCE_REPO_ID, config)
        for split in orig:
            orig_len = len(orig[split])
            try:
                our_len = len(all_data[config][split])
            except:
                our_len = "?"
            diff = orig_len - our_len if isinstance(our_len, int) else "?"
            if diff == 0:
                print(f"{config}/{split}: {our_len}")
            else:
                print(f"{config}/{split}: orig={orig_len}, ours={our_len}")
else:
    print("Не удалось перевести ни одной конфигурации")

In [ ]:
import os
import json
from datasets import DatasetDict, Dataset

def load_nested_dataset_dict(path):
    """Загружает DatasetDict, который может содержать внутри другие DatasetDict."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Путь {path} не существует")
    
    result = DatasetDict()
    # Проверяем, есть ли внешний dataset_dict.json
    if os.path.exists(os.path.join(path, "dataset_dict.json")):
        with open(os.path.join(path, "dataset_dict.json"), 'r') as f:
            splits = json.load(f)['splits']
        for split_name in splits:
            split_path = os.path.join(path, split_name)
            try:
                # Пробуем как DatasetDict (для конфигураций с несколькими сплитами)
                result[split_name] = DatasetDict.load_from_disk(split_path)
            except:
                # Иначе как обычный Dataset
                try:
                    result[split_name] = Dataset.load_from_disk(split_path)
                except:
                    pass
    else:
        # Может быть просто папка с датасетами?
        for item in os.listdir(path):
            item_path = os.path.join(path, item)
            if os.path.isdir(item_path):
                try:
                    result[item] = Dataset.load_from_disk(item_path)
                except:
                    pass
    return result

# Теперь загружаем
dataset = load_nested_dataset_dict("./iKAT_2023_ru")

# Удаляем служебные столбцы
for config_name, ds_dict_or_ds in dataset.items():
    if isinstance(ds_dict_or_ds, DatasetDict):
        for split_name, ds in ds_dict_or_ds.items():
            cols = [c for c in ['_index', '_failed', '_success'] if c in ds.column_names]
            if cols:
                dataset[config_name][split_name] = ds.remove_columns(cols)
                print(f"{config_name}/{split_name}: удалены {cols}")
    else:
        # Если это просто Dataset (на всякий случай)
        ds = ds_dict_or_ds
        cols = [c for c in ['_index', '_failed', '_success'] if c in ds.column_names]
        if cols:
            dataset[config_name] = ds.remove_columns(cols)
            print(f"{config_name} (единый): удалены {cols}")

# Сохраняем чистую версию
clean_path = "./iKAT_2023_ru_clean"
dataset.save_to_disk(clean_path)
print(f"\nЧистый датасет сохранён в {clean_path}")

In [ ]:
import os, json
from datasets import DatasetDict, Dataset
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")

CLEAN_PATH = "./iKAT_2023_ru_clean"
REPO_ID = "DeepPavlov/iKAT_2023_ru"

def load_nested_dataset_dict(path):
    """Загружает структуру датасета, где верхний уровень - папка с конфигурациями,
    а каждая конфигурация - DatasetDict (train/test)."""
    result = DatasetDict()
    for config_name in os.listdir(path):
        config_path = os.path.join(path, config_name)
        if not os.path.isdir(config_path):
            continue
        # Проверим, есть ли dataset_dict.json внутри (значит, это DatasetDict)
        if os.path.exists(os.path.join(config_path, "dataset_dict.json")):
            result[config_name] = DatasetDict.load_from_disk(config_path)
        else:
            # Возможно, это просто Dataset (например, если qrels скопирован как DatasetDict? маловероятно)
            # Но у нас все конфигурации - DatasetDict, так что этот вариант не понадобится
            pass
    return result

dataset = load_nested_dataset_dict(CLEAN_PATH)

for config_name, ds_dict in dataset.items():
    ds_dict.push_to_hub(
        REPO_ID,
        config_name=config_name,
        private=False,
        commit_message="Clean Russian translations of iKAT_2023"
    )
    print(f"{config_name} uploaded")

print(f"Готово: https://huggingface.co/datasets/{REPO_ID}")

In [ ]:
from datasets import load_dataset

try:
    ds = load_dataset("DeepPavlov/iKAT_2023_ru", "qrels", split="train")
    print(f"qrels загружен: {len(ds)} записей")
except Exception as e:
    print(f"Ошибка: {e}")

In [ ]:
import os, json
from datasets import load_dataset, DatasetDict, Dataset
from huggingface_hub import login, HfApi

# ---------- Авторизация ----------
login(token="YOUR_HF_TOKEN")   # замените на ваш токен

OLD_REPO_ID = "DeepPavlov/iKAT_2023_ru"
api = HfApi()

# ---------- Вспомогательная функция загрузки вложенных DatasetDict ----------
def load_nested_dataset_dict(path):
    """
    Загружает папку, где каждая подпапка (corpus, queries, qrels)
    сама является DatasetDict (содержит dataset_dict.json).
    """
    result = DatasetDict()
    if not os.path.isdir(path):
        raise FileNotFoundError(f"Папка {path} не найдена")
    for config_name in os.listdir(path):
        config_path = os.path.join(path, config_name)
        if not os.path.isdir(config_path):
            continue
        if os.path.exists(os.path.join(config_path, "dataset_dict.json")):
            result[config_name] = DatasetDict.load_from_disk(config_path)
    return result


# ---------- Загружаем переведённые конфигурации из локальной папки ----------
# Сначала пробуем _clean, если нет — обычную
for local_path in ["./iKAT_2023_ru_clean", "./iKAT_2023_ru"]:
    if os.path.isdir(local_path):
        print(f"Используем локальную папку: {local_path}")
        dataset = load_nested_dataset_dict(local_path)
        break
else:
    raise FileNotFoundError("Не найдена ни iKAT_2023_ru_clean, ни iKAT_2023_ru")

# Убедимся, что служебные столбцы удалены
for config in ['corpus', 'queries']:
    if config not in dataset:
        print(f"⚠️ Конфигурация {config} не найдена в локальной папке, пропускаем")
        continue
    ds_dict = dataset[config]
    for split in ds_dict:
        ds = ds_dict[split]
        cols_to_remove = [c for c in ['_index', '_failed', '_success'] if c in ds.column_names]
        if cols_to_remove:
            ds_dict[split] = ds.remove_columns(cols_to_remove)
            print(f"   Удалены столбцы {cols_to_remove} из {config}/{split}")

# ---------- Загрузка на Hugging Face ----------
# Пушим corpus и queries
for config in ['corpus', 'queries']:
    if config not in dataset:
        continue
    dataset[config].push_to_hub(
        OLD_REPO_ID,
        config_name=config,
        private=False,
        commit_message=f"Upload translated {config} (clean)"
    )
    print(f"{config} uploaded")

# Пушим оригинальный qrels (без перевода)
original_qrels = load_dataset("DeepPavlov/iKAT_2023", "qrels")
original_qrels.push_to_hub(
    OLD_REPO_ID,
    config_name="qrels",
    private=False,
    commit_message="Upload original qrels (viewer fix)"
)
print("qrels uploaded")

print(f"\nГотово! Проверьте: https://huggingface.co/datasets/{OLD_REPO_ID}")
print("Подождите 2-3 минуты, пока обновится вьювер.")